# Pipeline de Ingestão — Arquivos do Site BACEN

Lê os arquivos CSV baixados do site do BACEN (pasta `bacen_reclamacoes_site` no volume UC) e integra com os dados já extraídos via API.

**Classificação por conteúdo**: os CSVs são classificados pelas **colunas** (não pelo nome do arquivo), pois os nomes são inconsistentes.

**Fluxo**:
1. Descobre e classifica todos os CSVs do volume
2. **Reclamações (Bancos e Consórcios)** → compara com tabelas bronze existentes (API), marca redundantes
3. **Irregularidades (Bancos e Consórcios)** → cria novas tabelas bronze (complementar)
4. **Tabela de irregularidades** → cria tabela de referência
5. **Reclamações por conglomerado** → cria nova tabela com granularidade por instituição
6. Log de auditoria

**PDFs** (esclarecimentos, histórico) exigem modelo especial de extração — tratados como fase separada.

**Volume**: `/Volumes/workspace/bacen_bronze/bacen_reclamacoes_site/`


## Resumo do Pipeline de Ingestão — Arquivos do Site BACEN

### Objetivos

Este pipeline tem como objetivo **ingerir, classificar e integrar** os arquivos CSV disponibilizados no site do Banco Central do Brasil (BACEN) sobre reclamações e irregularidades no sistema financeiro, complementando os dados já extraídos via API. Por ser uma pipeline **apartada** (executada de forma independente da pipeline de API), todo o tratamento — desde a descoberta dos arquivos até a escrita nas tabelas finais — acontece aqui, abrangendo **todas as camadas do esquema medalhão**.

### O que foi feito

1. **Descoberta de arquivos**: varredura recursiva do volume UC (`bacen_reclamacoes_site`) em busca de todos os CSVs disponíveis.
2. **Classificação por conteúdo**: os arquivos são classificados pelo **conjunto de colunas** (não pelo nome do arquivo, que é inconsistente). Isso garante robustez mesmo com renomeações ou mudanças de padrão no site do BACEN.
3. **Sanitização**: nomes de colunas são normalizados para `snake_case` sem acentos, e metadados de auditoria (`_arquivo_origem`, `_data_carga`) são adicionados a cada registro.
4. **Deduplicação e comparação**: registros de reclamações (bancos e consórcios) são comparados com as tabelas bronze já existentes (originadas da API) para identificar redundâncias e dados complementares.
5. **Criação de novas tabelas**: irregularidades (bancos e consórcios), tabela de referência de irregularidades e reclamações por conglomerados são persistidas como novas tabelas.
6. **Log de auditoria**: todo o processamento é registrado em uma tabela de log, rastreando cada arquivo, seu tipo detectado e seu status.

### Fluxo geral

```
Volume UC (CSVs) → Descoberta → Classificação por colunas → Sanitização → Deduplicação/Comparação → Escrita nas tabelas → Log de auditoria → Verificação
```

### Camadas (Esquema Medalhão)

Como esta é uma pipeline apartada, **todas as camadas são tratadas neste notebook**:

| Camada | Papel neste pipeline | Tabelas / ações |
|---|---|---|
| **Bronze** | Ingestão bruta dos CSVs do site, com metadados de auditoria, sem transformações pesadas de negócio | `irregularidades_bancos`, `irregularidades_consorcios`, `tabela_irregularidades`, `reclamacoes_conglomerado`, `log_ingestao_site` |
| **Bronze (comparação)** | Reclamações de bancos e consórcios são comparadas com as tabelas bronze da API (`reclamacoes_bacen`, `consorcios_bacen`) para identificar redundância vs. complementaridade | Apenas análise — sem escrita de novos dados redundantes |
| **Silver / Gold** | *Não aplicadas aqui* — ficam a cargo de pipelines downstream que enriquecem, padronizam e agregam os dados bronze para consumo analítico | — |

> **Observação**: os arquivos **PDF** (esclarecimentos metodológicos, histórico de alterações) exigem modelo especial de extração e são tratados como fase separada, não coberta por este pipeline.

### Verificação final

Ao final da execução, o pipeline:

- Lista todas as tabelas criadas e confirma a contagem de registros em cada uma.
- Exibe uma amostra das irregularidades de bancos (ex.: 2025 T4) ordenada por quantidade de reclamações procedentes, validando que os dados foram persistidos corretamente.
- Registra no log de auditoria o resumo por **tipo detectado** e por **ano**, permitindo rastreabilidade completa da carga.

In [0]:
# ============================================================================
# CONFIGURAÇÃO E DESCOBERTA DE ARQUIVOS
# ============================================================================
# Define o volume UC de origem, catálogo/schema Bronze de destino e cria o
# schema se não existir. Varre recursivamente o volume em busca de todos os
# CSVs disponíveis, ordena e imprime o total encontrado (53 arquivos).
# ============================================================================
import pandas as pd
import os
import unicodedata
from datetime import datetime

VOLUME = "/Volumes/workspace/bacen_bronze/bacen_reclamacoes_site"
CATALOG = "workspace"
SCHEMA_BRONZE = "bacen_bronze"

# Criar schema se não existir
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}")

# Descobrir todos os arquivos CSV no volume
arquivos_csv = []
for root, dirs, files in os.walk(VOLUME):
    for f in files:
        if f.endswith('.csv'):
            arquivos_csv.append(os.path.join(root, f))

arquivos_csv.sort()
print(f"📁 {len(arquivos_csv)} arquivos CSV encontrados no volume")
for a in arquivos_csv[:5]:
    print(f"  {a}")
print(f"  ... ({len(arquivos_csv)} total)")

In [0]:
# ============================================================================
# CLASSIFICAÇÃO DOS CSVs POR CONTEÚDO (colunas), NÃO POR NOME
# ============================================================================
# Define a função classificar_csv() que lê as 3 primeiras linhas de cada
# CSV e identifica o tipo pelo conjunto de colunas (não pelo nome do
# arquivo, que é inconsistente). Tipos: irregularidades_bancos,
# irregularidades_consorcios, tabela_irregularidades_bancos,
# reclamacoes_bancos, reclamacoes_consorcios, reclamacoes_conglomerado.
# Itera sobre todos os arquivos, classifica e imprime um resumo por tipo.
# ============================================================================

def classificar_csv(caminho):
    """Classifica um CSV pelo conjunto de colunas. Retorna (tipo, df_preview)."""
    try:
        df = pd.read_csv(caminho, sep=';', dtype=str, encoding='latin-1', nrows=3)
        cols = set(c.strip() for c in df.columns if not str(c).startswith('Unnamed'))
        n_rows = len(pd.read_csv(caminho, sep=';', dtype=str, encoding='latin-1'))
    except Exception as e:
        return ('erro', None, 0, str(e))
    
    # Tabela de irregularidades: tem "Aplicação" e "Irregularidade" mas não tem instituição
    if 'Aplicação' in cols and 'Irregularidade' in cols and 'Instituição financeira' not in cols and 'Administradora de consórcio' not in cols:
        return ('tabela_irregularidades_bancos', df, n_rows, None)
    
    # Irregularidades por instituição (bancos): tem "Irregularidade" + "Instituição financeira"
    if 'Irregularidade' in cols and 'Instituição financeira' in cols:
        return ('irregularidades_bancos', df, n_rows, None)
    
    # Irregularidades por administradora (consórcios): tem "Irregularidade" + "Administradora de consórcio"
    if 'Irregularidade' in cols and 'Administradora de consórcio' in cols:
        return ('irregularidades_consorcios', df, n_rows, None)
    
    # Reclamações por conglomerado: tem "Conglomerado"
    if 'Conglomerado' in cols:
        return ('reclamacoes_conglomerado', df, n_rows, None)
    
    # Reclamações consórcios: tem "Semestre" + "Administradora de consórcio"
    if 'Semestre' in cols and 'Administradora de consórcio' in cols:
        return ('reclamacoes_consorcios', df, n_rows, None)
    
    # Reclamações bancos: tem "Trimestre" + "Instituição financeira" (sem "Irregularidade")
    if 'Trimestre' in cols and 'Instituição financeira' in cols and 'Irregularidade' not in cols:
        return ('reclamacoes_bancos', df, n_rows, None)
    
    return ('desconhecido', df, n_rows, f"Cols: {cols}")

# Classificar todos os arquivos
classificacao = []
for caminho in arquivos_csv:
    tipo, preview, n_rows, erro = classificar_csv(caminho)
    ano = caminho.split('/')[-2]  # pasta do ano
    nome = os.path.basename(caminho)
    classificacao.append({
        'caminho': caminho,
        'nome': nome,
        'ano_pasta': ano,
        'tipo': tipo,
        'linhas': n_rows,
        'erro': erro
    })
    status = '✅' if tipo != 'desconhecido' else '❓'
    print(f"  {status} [{tipo}] {ano}/{nome} ({n_rows} linhas)")

# Resumo
df_class = pd.DataFrame(classificacao)
print(f"\n📊 Resumo de classificação:")
print(df_class.groupby('tipo').agg(arquivos=('nome', 'count'), linhas=('linhas', 'sum')).to_string())

# Verificar se há arquivos não classificados
desconhecidos = df_class[df_class['tipo'] == 'desconhecido']
if len(desconhecidos) > 0:
    print(f"\n⚠️ {len(desconhecidos)} arquivos não classificados:")
    for _, r in desconhecidos.iterrows():
        print(f"  {r['nome']}: {r['erro']}")

In [0]:
# ============================================================================
# FUNÇÃO AUXILIAR: SANITIZAR NOMES DE COLUNAS
# ============================================================================
# Define duas funções reutilizáveis: sanitizar_nome() converte nomes de
# colunas PascalCase com acentos para snake_case ASCII, e ler_csv_completo()
# lê um CSV completo, remove colunas Unnamed, sanitiza nomes e adiciona
# metadados de auditoria (_arquivo_origem, _data_carga).
# ============================================================================

def sanitizar_nome(nome):
    """Converte nome de coluna para snake_case sem acentos."""
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')
    nome = nome.replace(' - ', '_').replace(' ', '_').replace('-', '_')
    nome = ''.join(c if c.isalnum() or c == '_' else '_' for c in nome)
    while '__' in nome:
        nome = nome.replace('__', '_')
    return nome.strip('_').lower()

def ler_csv_completo(caminho):
    """Lê um CSV completo, remove colunas Unnamed e sanitiza nomes."""
    df = pd.read_csv(caminho, sep=';', dtype=str, encoding='latin-1')
    cols = [c for c in list(df) if not str(c).startswith('Unnamed')]
    df = df[cols].rename(columns={c: sanitizar_nome(c) for c in cols})
    # Adicionar metadados
    df['_arquivo_origem'] = os.path.basename(caminho)
    df['_data_carga'] = datetime.now().isoformat()
    return df

print("✅ Funções auxiliares definidas")

In [0]:
# ============================================================================
# 1. RECLAMAÇÕES BANCOS — COMPARAR COM TABELA EXISTENTE (API)
# ============================================================================
# Lê todos os CSVs classificados como reclamacoes_bancos, consolida em
# um DataFrame e compara com a tabela Bronze já existente (originada da
# API) por (ano, trimestre, instituição). Identifica quantos registros
# do site são redundantes vs complementares. Resultado: 100% redundantes.
# ============================================================================

arquivos_rec_bancos = [r['caminho'] for r in classificacao if r['tipo'] == 'reclamacoes_bancos']
print(f"📋 {len(arquivos_rec_bancos)} arquivos de Reclamações (Bancos) para processar")

if arquivos_rec_bancos:
    # Ler e consolidar todos os CSVs de reclamações bancos
    lista_dfs = [ler_csv_completo(c) for c in arquivos_rec_bancos]
    df_site = pd.concat(lista_dfs, ignore_index=True)
    print(f"   Total de linhas nos arquivos do site: {len(df_site)}")
    
    # Normalizar trimestre: "1º" → 1, "2º" → 2, etc.
    if 'trimestre' in df_site.columns:
        df_site['trimestre_num'] = df_site['trimestre'].str.extract(r'(\d+)').astype(int)
    
    # Carregar tabela existente (API)
    df_api = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.reclamacoes_bacen").toPandas()
    print(f"   Total de linhas na tabela API (bronze): {len(df_api)}")
    
    # Comparar por ano + trimestre + instituição
    df_site_key = df_site[['ano', 'trimestre_num', 'instituicao_financeira']].copy()
    df_site_key.columns = ['ano', 'trimestre', 'instituicao']
    df_api_key = df_api[['ano', 'periodo_extracao', 'instituicao_financeira']].copy()
    df_api_key.columns = ['ano', 'trimestre', 'instituicao']
    df_api_key['ano'] = df_api_key['ano'].astype(str)
    df_api_key['trimestre'] = df_api_key['trimestre'].astype(int)
    df_site_key['ano'] = df_site_key['ano'].astype(str)
    
    # Merge para identificar redundantes vs complementares
    df_site_key['_source'] = 'site'
    df_api_key['_source'] = 'api'
    merged = df_site_key.merge(df_api_key, on=['ano', 'trimestre', 'instituicao'], how='left', indicator=True)
    
    redundantes = (merged['_merge'] == 'both').sum()
    complementares = (merged['_merge'] == 'left_only').sum()
    print(f"\n   🔴 Redundantes (já na API): {redundantes}")
    print(f"   🟢 Complementares (novos): {complementares}")
    
    if complementares > 0:
        novos = merged[merged['_merge'] == 'left_only'][['ano', 'trimestre', 'instituicao']]
        print(f"   Novos registros:")
        print(novos.groupby(['ano', 'trimestre']).size().to_string())
    else:
        print(f"   ✅ Todos os registros do site já estão na tabela API — dados 100% redundantes")
else:
    print("   Nenhum arquivo de Reclamações (Bancos) encontrado")

In [0]:
# ============================================================================
# 2. RECLAMAÇÕES CONSÓRCIOS — COMPARAR COM TABELA EXISTENTE (API)
# ============================================================================
# Lê os CSVs de reclamacoes_consorcios, consolida e compara com a tabela
# Bronze existente (API) por (ano, semestre, administradora). Mesma
# lógica da célula anterior: identifica redundantes vs complementares.
# Resultado: 100% redundantes.
# ============================================================================

arquivos_rec_cons = [r['caminho'] for r in classificacao if r['tipo'] == 'reclamacoes_consorcios']
print(f"📋 {len(arquivos_rec_cons)} arquivos de Reclamações (Consórcios) para processar")

if arquivos_rec_cons:
    lista_dfs = [ler_csv_completo(c) for c in arquivos_rec_cons]
    df_site = pd.concat(lista_dfs, ignore_index=True)
    print(f"   Total de linhas nos arquivos do site: {len(df_site)}")
    
    # Normalizar semestre
    if 'semestre' in df_site.columns:
        df_site['semestre_num'] = df_site['semestre'].str.extract(r'(\d+)').astype(int)
    
    # Carregar tabela existente
    df_api = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.consorcios_bacen").toPandas()
    print(f"   Total de linhas na tabela API (bronze): {len(df_api)}")
    
    # Comparar por ano + semestre + administradora
    df_site_key = df_site[['ano', 'semestre_num', 'administradora_de_consorcio']].copy()
    df_site_key.columns = ['ano', 'semestre', 'administradora']
    df_api_key = df_api[['ano', '_periodo_extracao', 'administradora_de_consorcio']].copy()
    df_api_key.columns = ['ano', 'semestre', 'administradora']
    df_api_key['ano'] = df_api_key['ano'].astype(str)
    df_api_key['semestre'] = df_api_key['semestre'].astype(int)
    df_site_key['ano'] = df_site_key['ano'].astype(str)
    
    merged = df_site_key.merge(df_api_key, on=['ano', 'semestre', 'administradora'], how='left', indicator=True)
    redundantes = (merged['_merge'] == 'both').sum()
    complementares = (merged['_merge'] == 'left_only').sum()
    print(f"\n   🔴 Redundantes (já na API): {redundantes}")
    print(f"   🟢 Complementares (novos): {complementares}")
    
    if complementares > 0:
        novos = merged[merged['_merge'] == 'left_only'][['ano', 'semestre', 'administradora']]
        print(f"   Novos registros:")
        print(novos.groupby(['ano', 'semestre']).size().to_string())
    else:
        print(f"   ✅ Todos os registros do site já estão na tabela API — dados 100% redundantes")
else:
    print("   Nenhum arquivo de Reclamações (Consórcios) encontrado")

In [0]:
# ============================================================================
# 3. IRREGULARIDADES POR INSTITUIÇÃO (BANCOS) — NOVA TABELA BRONZE
# ============================================================================
# Lê todos os CSVs de irregularidades_bancos, consolida, normaliza
# trimestre ("1º" → 1) e deduplica por (ano, trimestre, instituição,
# irregularidade) mantendo o registro de _data_carga mais recente.
# Salva como tabela Delta Bronze. Cobertura: 2024T1 a 2026T2, ~42.106 registros.
# ============================================================================

arquivos_irr_bancos = [r['caminho'] for r in classificacao if r['tipo'] == 'irregularidades_bancos']
print(f"📋 {len(arquivos_irr_bancos)} arquivos de Irregularidades (Bancos)")

if arquivos_irr_bancos:
    lista_dfs = [ler_csv_completo(c) for c in arquivos_irr_bancos]
    df_irr_bancos = pd.concat(lista_dfs, ignore_index=True)
    
    # Normalizar trimestre
    if 'trimestre' in df_irr_bancos.columns:
        df_irr_bancos['trimestre_num'] = df_irr_bancos['trimestre'].str.extract(r'(\d+)').astype(int)
    
    periodos_unicos = sorted((df_irr_bancos['ano'].astype(str) + 'T' + df_irr_bancos['trimestre_num'].astype(str)).unique())
    print(f"   Total: {len(df_irr_bancos)} linhas | Colunas: {list(df_irr_bancos.columns)}")
    print(f"   Períodos: {periodos_unicos}")
    
    # Deduplicar: manter apenas o registro mais recente por (ano, trimestre, instituicao, irregularidade)
    df_irr_bancos = df_irr_bancos.sort_values('_data_carga', ascending=False).drop_duplicates(
        subset=['ano', 'trimestre_num', 'instituicao_financeira', 'irregularidade'],
        keep='first'
    )
    print(f"   Após dedup: {len(df_irr_bancos)} linhas")
    
    # Salvar como tabela bronze
    TABELA = f"{CATALOG}.{SCHEMA_BRONZE}.irregularidades_bancos"
    spark.createDataFrame(df_irr_bancos).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA)
    print(f"   ✅ Tabela criada: {TABELA} | {len(df_irr_bancos)} registros")
else:
    print("   Nenhum arquivo encontrado")

In [0]:
# ============================================================================
# 4. IRREGULARIDADES POR ADMINISTRADORA (CONSÓRCIOS) — NOVA TABELA BRONZE
# ============================================================================
# Lê os CSVs de irregularidades_consorcios, consolida, normaliza semestre
# e deduplica por (ano, semestre, administradora, irregularidade) mantendo
# o registro mais recente. Salva como tabela Delta Bronze. ~2.738 registros.
# ============================================================================

arquivos_irr_cons = [r['caminho'] for r in classificacao if r['tipo'] == 'irregularidades_consorcios']
print(f"📋 {len(arquivos_irr_cons)} arquivos de Irregularidades (Consórcios)")

if arquivos_irr_cons:
    lista_dfs = [ler_csv_completo(c) for c in arquivos_irr_cons]
    df_irr_cons = pd.concat(lista_dfs, ignore_index=True)
    
    # Normalizar semestre
    if 'semestre' in df_irr_cons.columns:
        df_irr_cons['semestre_num'] = df_irr_cons['semestre'].str.extract(r'(\d+)').astype(int)
    
    print(f"   Total: {len(df_irr_cons)} linhas | Colunas: {list(df_irr_cons.columns)}")
    
    # Deduplicar
    df_irr_cons = df_irr_cons.sort_values('_data_carga', ascending=False).drop_duplicates(
        subset=['ano', 'semestre_num', 'administradora_de_consorcio', 'irregularidade'],
        keep='first'
    )
    print(f"   Após dedup: {len(df_irr_cons)} linhas")
    
    TABELA = f"{CATALOG}.{SCHEMA_BRONZE}.irregularidades_consorcios"
    spark.createDataFrame(df_irr_cons).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA)
    print(f"   ✅ Tabela criada: {TABELA} | {len(df_irr_cons)} registros")
else:
    print("   Nenhum arquivo encontrado")

In [0]:
# ============================================================================
# 5. TABELA DE IRREGULARIDADES — TABELA DE REFERÊNCIA
# ============================================================================
# Lê os CSVs classificados como tabela_irregularidades_bancos, que
# contêm o catálogo de irregularidades com descrição, tipo e aplicação.
# Normaliza trimestre e semestre (Int64 nullable) e deduplica por
# (ano, período, tipo, irregularidade). ~1.800 registros após dedup.
# ============================================================================

arquivos_tab_irr = [r['caminho'] for r in classificacao if r['tipo'] == 'tabela_irregularidades_bancos']
print(f"📋 {len(arquivos_tab_irr)} arquivos de Tabela de Irregularidades")

if arquivos_tab_irr:
    lista_dfs = [ler_csv_completo(c) for c in arquivos_tab_irr]
    df_tab_irr = pd.concat(lista_dfs, ignore_index=True)
    
    # Normalizar trimestre (se aplicável) — usar Int64 nullable pois consórcios usam Semestre
    if 'trimestre' in df_tab_irr.columns:
        df_tab_irr['trimestre_num'] = df_tab_irr['trimestre'].str.extract(r'(\d+)').astype('Int64')
    # Normalizar semestre (consórcios)
    if 'semestre' in df_tab_irr.columns:
        df_tab_irr['semestre_num'] = df_tab_irr['semestre'].str.extract(r'(\d+)').astype('Int64')
    
    print(f"   Total: {len(df_tab_irr)} linhas | Colunas: {list(df_tab_irr.columns)}")
    
    # Deduplicar por (ano, trimestre, tipo, irregularidade)
    dedup_cols = [c for c in ['ano', 'trimestre_num', 'tipo', 'irregularidade'] if c in df_tab_irr.columns]
    df_tab_irr = df_tab_irr.sort_values('_data_carga', ascending=False).drop_duplicates(subset=dedup_cols, keep='first')
    print(f"   Após dedup: {len(df_tab_irr)} linhas")
    
    TABELA = f"{CATALOG}.{SCHEMA_BRONZE}.tabela_irregularidades"
    spark.createDataFrame(df_tab_irr).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA)
    print(f"   ✅ Tabela criada: {TABELA} | {len(df_tab_irr)} registros")
else:
    print("   Nenhum arquivo encontrado")

In [0]:
# ============================================================================
# 6. RECLAMAÇÕES POR CONGLOMERADO — NOVA TABELA BRONZE
# ============================================================================
# Lê os CSVs de reclamacoes_conglomerado, que têm granularidade por
# conglomerado (não por instituição individual). Normaliza trimestre,
# deduplica por (ano, trimestre, cnpj_if, instituição) e salva como
# tabela Delta Bronze. ~2.030 registros após dedup.
# ============================================================================

arquivos_congl = [r['caminho'] for r in classificacao if r['tipo'] == 'reclamacoes_conglomerado']
print(f"📋 {len(arquivos_congl)} arquivos de Reclamações por Conglomerado")

if arquivos_congl:
    lista_dfs = [ler_csv_completo(c) for c in arquivos_congl]
    df_congl = pd.concat(lista_dfs, ignore_index=True)
    
    # Normalizar trimestre
    if 'trimestre' in df_congl.columns:
        df_congl['trimestre_num'] = df_congl['trimestre'].str.extract(r'(\d+)').astype(int)
    
    print(f"   Total: {len(df_congl)} linhas | Colunas: {list(df_congl.columns)}")
    
    # Deduplicar por (ano, trimestre, cnpj_if, instituicao_financeira)
    df_congl = df_congl.sort_values('_data_carga', ascending=False).drop_duplicates(
        subset=['ano', 'trimestre_num', 'cnpj_if', 'instituicao_financeira'],
        keep='first'
    )
    print(f"   Após dedup: {len(df_congl)} linhas")
    
    TABELA = f"{CATALOG}.{SCHEMA_BRONZE}.reclamacoes_conglomerado"
    spark.createDataFrame(df_congl).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA)
    print(f"   ✅ Tabela criada: {TABELA} | {len(df_congl)} registros")
else:
    print("   Nenhum arquivo encontrado")

In [0]:
# ============================================================================
# 7. LOG DE AUDITORIA — REGISTRA PROCESSAMENTO
# ============================================================================
# Constrói um DataFrame de log com um registro por arquivo processado:
# nome, ano da pasta, tipo detectado, número de linhas, status e timestamp.
# Salva como tabela Delta Bronze (log_ingestao_site) para rastreabilidade.
# ============================================================================

log_entries = []
for r in classificacao:
    log_entries.append({
        'arquivo': r['nome'],
        'ano_pasta': r['ano_pasta'],
        'tipo_detectado': r['tipo'],
        'linhas': r['linhas'],
        'status': 'processado' if r['tipo'] != 'desconhecido' else 'nao_classificado',
        'data_processamento': datetime.now().isoformat()
    })

df_log = pd.DataFrame(log_entries)
TABELA_LOG = f"{CATALOG}.{SCHEMA_BRONZE}.log_ingestao_site"
spark.createDataFrame(df_log).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA_LOG)

print(f"📋 Log de auditoria salvo: {TABELA_LOG}")
print(f"   {len(df_log)} registros processados")
print(f"\n   Resumo por tipo:")
print(df_log.groupby(['tipo_detectado', 'status']).agg(
    arquivos=('arquivo', 'count'),
    linhas=('linhas', 'sum')
).to_string())
print(f"\n   Resumo por ano:")
print(df_log.groupby('ano_pasta').agg(
    arquivos=('arquivo', 'count'),
    linhas=('linhas', 'sum')
).to_string())

In [0]:
# ============================================================================
# 8. VERIFICAÇÃO DAS NOVAS TABELAS
# ============================================================================
# Itera sobre as 5 tabelas Bronze criadas (irregularidades_bancos,
# irregularidades_consorcios, tabela_irregularidades, reclamacoes_conglomerado,
# log_ingestao_site) e imprime a contagem de registros de cada uma. Exibe
# também uma amostra das irregularidades de bancos (2025 T4) ordenada por
# quantidade de reclamações procedentes.
# ============================================================================

print("=" * 60)
print("VERIFICAÇÃO DAS TABELAS CRIADAS")
print("=" * 60)

tabelas_novas = [
    'irregularidades_bancos',
    'irregularidades_consorcios',
    'tabela_irregularidades',
    'reclamacoes_conglomerado',
    'log_ingestao_site'
]

for tab in tabelas_novas:
    fqn = f"{CATALOG}.{SCHEMA_BRONZE}.{tab}"
    try:
        count = spark.sql(f"SELECT COUNT(*) FROM {fqn}").collect()[0][0]
        print(f"  ✅ {fqn}: {count} registros")
    except Exception as e:
        print(f"  ❌ {fqn}: {e}")

# Amostra das irregularidades (bancos) — os dados mais ricos
print(f"\n📊 Amostra: Irregularidades Bancos (2025 T4)")
try:
    spark.sql(f"""
    SELECT instituicao_financeira, irregularidade, 
           quantidade_de_reclamacoes_procedentes AS qtd
    FROM {CATALOG}.{SCHEMA_BRONZE}.irregularidades_bancos
    WHERE ano = '2025' AND trimestre_num = 4
    ORDER BY CAST(quantidade_de_reclamacoes_procedentes AS INT) DESC
    LIMIT 10
    """).show(truncate=40)
except Exception as e:
    print(f"  Erro: {e}")

## PDFs — Fase Separada

Os arquivos PDF no volume (`Historico+de+alteracoes+da+tabela+de+irregularidades.pdf`, 
`Ranking+mensal+-+Esclarecimentos+sobre+os+arquivos+de+download.pdf`, 
`Ranking+semestral+-+Esclarecimentos+sobre+os+arquivos+de+download.pdf`)
contêm documentação sobre metodologia e mudanças nas tabelas de irregularidades.

**Próximos passos para PDFs**:
1. Usar `ai_parse_document` (Agent Bricks tile) para extrair texto estruturado
2. Ou usar a função SQL `ai_extract` para puxar informações específicas
3. Armazenar resumos em uma tabela `bacen_bronze.documentacao_pdfs`

Estes PDFs não contêm dados tabulares — são textos explicativos. A extração
requer um modelo de linguagem para identificar informações pertinentes 
(ex: mudanças de categoria, novas irregularidades, alterações metodológicas).

## Camadas Silver e Gold — Dados Complementares do Site

Transforma as tabelas bronze criadas a partir dos CSVs do site em tabelas Silver (limpeza e tipos)
e Gold (tabelas analíticas com rankings e resumos).

**Silver**: irregularidades_bancos, irregularidades_consorcios, reclamacoes_conglomerado, tabela_irregularidades
**Gold**: ranking_irregularidades_bancos, resumo_irregularidades_anual, top_irregularidades_periodo, ranking_irregularidades_consorcios

In [0]:
%sql
-- ============================================================================
-- SILVER: IRREGULARIDADES BANCOS — LIMPEZA E TIPOS
-- COALESCE unifica colunas de formato antigo (2024T3+) e novo (2024T1)
-- ============================================================================
-- CTAS da Silver a partir da Bronze raw. Aplica CAST de strings para INT
-- (trimestre, contagens), cria categoria (Top/Demais) via CASE e COALESCE
-- para unificar colunas de formato antigo e novo (mudança de schema no
-- site do BACEN). TRIM e NULLIF em texto. Renomeia colunas longas para
-- nomes curtos (qtd_procedentes, qtd_outras, etc.).
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.irregularidades_bancos AS
SELECT
  CAST(ano AS INT) AS ano,
  CAST(trimestre_num AS INT) AS trimestre,
  CASE WHEN categoria LIKE 'Top%' THEN 'Top' ELSE 'Demais' END AS categoria,
  TRIM(tipo) AS tipo,
  NULLIF(TRIM(cnpj_if), '') AS cnpj_if,
  TRIM(instituicao_financeira) AS instituicao_financeira,
  TRIM(irregularidade) AS irregularidade,
  -- Unificar colunas de quantidade procedentes (formato antigo vs novo)
  CAST(COALESCE(
    NULLIF(TRIM(quantidade_de_reclamacoes_procedentes), ''),
    NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes), '')
  ) AS LONG) AS qtd_procedentes,
  -- Outras colunas (formato novo, podem ser NULL em períodos antigos)
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_outras), '') AS LONG) AS qtd_outras,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_nao_reguladas), '') AS LONG) AS qtd_nao_reguladas,
  CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes), '') AS LONG) AS qtd_total,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes_extrapoladas), '') AS LONG) AS qtd_procedentes_extrapoladas,
  TRIM(_arquivo_origem) AS arquivo_origem,
  CAST(_data_carga AS TIMESTAMP) AS data_carga
FROM workspace.bacen_bronze.irregularidades_bancos

In [0]:
%sql
-- ============================================================================
-- SILVER: IRREGULARIDADES CONSÓRCIOS — LIMPEZA E TIPOS
-- ============================================================================
-- CTAS da Silver a partir da Bronze raw. Aplica CAST para INT (semestre,
-- contagens) e DECIMAL (índice), TRIM em texto e renomeação de colunas
-- longas para nomes curtos. Mesma lógica da Silver de bancos, adaptada
-- para a estrutura semestral dos consórcios.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.irregularidades_consorcios AS
SELECT
  CAST(ano AS INT) AS ano,
  CAST(semestre_num AS INT) AS semestre,
  NULLIF(TRIM(cnpj_ac), '') AS cnpj_ac,
  TRIM(administradora_de_consorcio) AS administradora,
  TRIM(irregularidade) AS irregularidade,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_procedentes), '') AS LONG) AS qtd_procedentes,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_reguladas_outras), '') AS LONG) AS qtd_outras,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_nao_reguladas), '') AS LONG) AS qtd_nao_reguladas,
  CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes), '') AS LONG) AS qtd_total,
  TRIM(_arquivo_origem) AS arquivo_origem,
  CAST(_data_carga AS TIMESTAMP) AS data_carga
FROM workspace.bacen_bronze.irregularidades_consorcios

In [0]:
%sql
-- ============================================================================
-- SILVER: RECLAMAÇÕES POR CONGLOMERADO — LIMPEZA E TIPOS
-- ============================================================================
-- CTAS da Silver a partir da Bronze raw. Aplica CAST para INT (trimestre,
-- contagens) e DECIMAL (índice), cria categoria Top/Demais via CASE, TRIM
-- em texto e renomeação de colunas. Mantém a granularidade por conglomerado.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.reclamacoes_conglomerado AS
SELECT
  CAST(ano AS INT) AS ano,
  CAST(trimestre_num AS INT) AS trimestre,
  CASE WHEN categoria LIKE 'Top%' THEN 'Top' ELSE 'Demais' END AS categoria,
  TRIM(tipo) AS tipo,
  NULLIF(TRIM(cnpj_if), '') AS cnpj_if,
  TRIM(conglomerado) AS conglomerado,
  TRIM(instituicao_financeira) AS instituicao_financeira,
  CAST(REPLACE(REPLACE(NULLIF(TRIM(indice), ''), '.', ''), ',', '.') AS DECIMAL(10, 2)) AS indice,
  CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes_respondidas), '') AS LONG) AS qtd_total_respondidas,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_procedentes), '') AS LONG) AS qtd_procedentes,
  CAST(NULLIF(TRIM(quantidade_de_reclamacoes_procedentes_extrapoladas), '') AS LONG) AS qtd_procedentes_extrapoladas,
  CAST(NULLIF(TRIM(quantidade_total_de_reclamacoes_analisadas), '') AS LONG) AS qtd_analisadas,
  TRIM(_arquivo_origem) AS arquivo_origem,
  CAST(_data_carga AS TIMESTAMP) AS data_carga
FROM workspace.bacen_bronze.reclamacoes_conglomerado

In [0]:
%sql
-- ============================================================================
-- SILVER: TABELA DE IRREGULARIDADES — REFERÊNCIA LIMPA
-- ============================================================================
-- CTAS da Silver a partir da Bronze raw. Aplica CAST para INT (ano,
-- período), COALESCE entre trimestre e semestre (bancos usam T, consórcios
-- usam S), cria tipo_periodo ('T' ou 'S') e TRIM em texto. Filtra linhas
-- com irregularidade vazia. Tabela de referência para consulta de tipos.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_silver.tabela_irregularidades AS
SELECT
  CAST(ano AS INT) AS ano,
  COALESCE(CAST(trimestre_num AS INT), CAST(semestre_num AS INT)) AS periodo,
  CASE WHEN trimestre_num IS NOT NULL THEN 'T' ELSE 'S' END AS tipo_periodo,
  TRIM(tipo) AS tipo,
  TRIM(irregularidade) AS irregularidade,
  TRIM(aplicacao) AS aplicacao
FROM workspace.bacen_bronze.tabela_irregularidades
WHERE TRIM(irregularidade) IS NOT NULL
  AND TRIM(irregularidade) != ''

In [0]:
%sql
-- ============================================================================
-- GOLD 1: RANKING DE IRREGULARIDADES POR INSTITUIÇÃO (BANCOS)
-- Ranking por volume de reclamações procedentes por irregularidade
-- ============================================================================
-- CTAS da Gold a partir da Silver. Preserva todas as colunas e adiciona
-- dois rankings com ROW_NUMBER(): ranking_dentro_instituicao (por
-- qtd_procedentes DESC dentro de cada instituição) e
-- ranking_dentro_irregularidade (por qtd_procedentes DESC dentro de cada
-- tipo de irregularidade). Tabela de granularidade máxima.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.ranking_irregularidades_bancos AS
SELECT
  ano, trimestre, categoria, tipo, cnpj_if, instituicao_financeira, irregularidade,
  qtd_procedentes, qtd_outras, qtd_nao_reguladas, qtd_total,
  ROW_NUMBER() OVER (
    PARTITION BY ano, trimestre, instituicao_financeira
    ORDER BY qtd_procedentes DESC NULLS LAST
  ) AS ranking_dentro_instituicao,
  ROW_NUMBER() OVER (
    PARTITION BY ano, trimestre, irregularidade
    ORDER BY qtd_procedentes DESC NULLS LAST
  ) AS ranking_dentro_irregularidade,
  data_carga
FROM workspace.bacen_silver.irregularidades_bancos

In [0]:
%sql
-- ============================================================================
-- GOLD 2: RESUMO ANUAL DE IRREGULARIDADES (BANCOS)
-- ============================================================================
-- CTAS da Gold a partir da Silver. Agrega por (ano, categoria) com
-- COUNT(DISTINCT) de instituições e tipos de irregularidade, e SUM de
-- procedentes e total de reclamações. Usada no dashboard para a evolução
-- temporal por categoria (Top/Demais).
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.resumo_irregularidades_anual AS
SELECT
  ano,
  categoria,
  COUNT(DISTINCT instituicao_financeira) AS qtd_instituicoes,
  COUNT(DISTINCT irregularidade) AS qtd_tipos_irregularidade,
  SUM(qtd_procedentes) AS total_procedentes,
  SUM(qtd_total) AS total_reclamacoes
FROM workspace.bacen_silver.irregularidades_bancos
GROUP BY ano, categoria
ORDER BY ano, categoria

In [0]:
%sql
-- ============================================================================
-- GOLD 3: TOP IRREGULARIDADES POR PERÍODO (BANCOS)
-- ============================================================================
-- CTAS da Gold a partir da Silver. CTE por_irregularidade agrega por
-- (ano, trimestre, irregularidade) com SUM de procedentes e COUNT
-- DISTINCT de instituições. CTE ranked atribui rank_periodo com
-- ROW_NUMBER() por período. Filtra top 10 por período. Usada no
-- dashboard para o ranking e super categorias de irregularidades.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.top_irregularidades_periodo AS
WITH por_irregularidade AS (
  SELECT
    ano, trimestre, irregularidade,
    COUNT(DISTINCT instituicao_financeira) AS qtd_instituicoes,
    SUM(qtd_procedentes) AS total_procedentes,
    SUM(qtd_total) AS total_reclamacoes
  FROM workspace.bacen_silver.irregularidades_bancos
  GROUP BY ano, trimestre, irregularidade
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY ano, trimestre ORDER BY total_procedentes DESC NULLS LAST) AS rank_periodo
  FROM por_irregularidade
)
SELECT ano, trimestre, irregularidade, qtd_instituicoes, total_procedentes, total_reclamacoes, rank_periodo
FROM ranked
WHERE rank_periodo <= 10
ORDER BY ano, trimestre, rank_periodo

In [0]:
%sql
-- ============================================================================
-- GOLD 4: RANKING DE IRREGULARIDADES (CONSÓRCIOS)
-- ============================================================================
-- CTAS da Gold a partir da Silver de consórcios. Mesma lógica do Gold 1
-- mas adaptada para a estrutura semestral: dois rankings com
-- ROW_NUMBER() — ranking_dentro_administradora e ranking_dentro_irregularidade,
-- particionados por (ano, semestre).
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.ranking_irregularidades_consorcios AS
SELECT
  ano, semestre, cnpj_ac, administradora, irregularidade,
  qtd_procedentes, qtd_outras, qtd_nao_reguladas, qtd_total,
  ROW_NUMBER() OVER (
    PARTITION BY ano, semestre, administradora
    ORDER BY qtd_procedentes DESC NULLS LAST
  ) AS ranking_dentro_administradora,
  ROW_NUMBER() OVER (
    PARTITION BY ano, semestre, irregularidade
    ORDER BY qtd_procedentes DESC NULLS LAST
  ) AS ranking_dentro_irregularidade,
  data_carga
FROM workspace.bacen_silver.irregularidades_consorcios

In [0]:
%sql
-- ============================================================================
-- GOLD 5: EVOLUÇÃO DE IRREGULARIDADES POR INSTITUIÇÃO (BANCOS)
-- Pivot: total de procedentes por instituição ao longo dos trimestres
-- ============================================================================
-- CTAS da Gold a partir da Silver. Agrega por (instituição, ano, trimestre)
-- com SUM de procedentes e total, e COUNT DISTINCT de tipos de
-- irregularidade. Tabela de série temporal por instituição, usada no
-- dashboard para o gráfico de Top Instituições (período mais recente).
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.evolucao_irregularidades_instituicao AS
SELECT
  instituicao_financeira,
  ano, trimestre,
  SUM(qtd_procedentes) AS total_procedentes,
  SUM(qtd_total) AS total_reclamacoes,
  COUNT(DISTINCT irregularidade) AS qtd_tipos_irregularidade
FROM workspace.bacen_silver.irregularidades_bancos
GROUP BY instituicao_financeira, ano, trimestre

In [0]:
%sql
-- ============================================================================
-- GOLD 6: RESUMO ANUAL DE IRREGULARIDADES (CONSÓRCIOS)
-- ============================================================================
-- CTAS da Gold a partir da Silver de consórcios. Agrega por ano com
-- COUNT(DISTINCT) de administradoras e tipos de irregularidade, e SUM
-- de procedentes e total de reclamações. Equivalente consórcios do Gold 2.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_gold.resumo_irregularidades_consorcios_anual AS
SELECT
  ano,
  COUNT(DISTINCT administradora) AS qtd_administradoras,
  COUNT(DISTINCT irregularidade) AS qtd_tipos_irregularidade,
  SUM(qtd_procedentes) AS total_procedentes,
  SUM(qtd_total) AS total_reclamacoes
FROM workspace.bacen_silver.irregularidades_consorcios
GROUP BY ano
ORDER BY ano

In [0]:
# ============================================================================
# VERIFICAÇÃO SILVER E GOLD
# ============================================================================
# Itera sobre as 4 tabelas Silver e 6 tabelas Gold criadas neste notebook,
# imprimindo a contagem de registros de cada uma. Finaliza com uma amostra
# do Top 5 Irregularidades de 2025 T4 (rank_periodo, irregularidade,
# qtd_instituicoes, total_procedentes) para validação visual.
# ============================================================================
print("=" * 70)
print("VERIFICAÇÃO DAS CAMADAS SILVER E GOLD")
print("=" * 70)

print("\n🥈 CAMADA SILVER")
for tab in ['irregularidades_bancos', 'irregularidades_consorcios', 'reclamacoes_conglomerado', 'tabela_irregularidades']:
    c = spark.sql(f"SELECT COUNT(*) FROM workspace.bacen_silver.{tab}").collect()[0][0]
    print(f"  ✅ workspace.bacen_silver.{tab}: {c:,} registros")

print("\n🥇 CAMADA GOLD")
for tab in ['ranking_irregularidades_bancos', 'resumo_irregularidades_anual', 'top_irregularidades_periodo', 'ranking_irregularidades_consorcios', 'evolucao_irregularidades_instituicao', 'resumo_irregularidades_consorcios_anual']:
    c = spark.sql(f"SELECT COUNT(*) FROM workspace.bacen_gold.{tab}").collect()[0][0]
    print(f"  ✅ workspace.bacen_gold.{tab}: {c:,} registros")

print("\n📊 Top 5 Irregularidades — 2025 T4")
spark.sql("""
SELECT rank_periodo, irregularidade, qtd_instituicoes, total_procedentes
FROM workspace.bacen_gold.top_irregularidades_periodo
WHERE ano = 2025 AND trimestre = 4
ORDER BY rank_periodo LIMIT 5
""").show(truncate=50)

## Extração de PDFs com `ai_parse_document`

Os PDFs no volume contêm documentação metodológica do BACEN:
1. **Histórico de alterações da tabela de irregularidades** — cronologia de mudanças
2. **Ranking mensal — Esclarecimentos** — explicações sobre arquivos de download mensal
3. **Ranking semestral — Esclarecimentos** — explicações sobre arquivos de download semestral

Pipeline: `READ_FILES(binaryFile)` → `ai_parse_document(v2)` → `ai_extract(v2.1)` → tabela UC

In [0]:
%sql
-- ============================================================================
-- EXTRAÇÃO DE PDFs: ai_parse_document + ai_extract
-- ============================================================================
-- Lê PDFs do volume com READ_FILES(binaryFile), processa com
-- ai_parse_document (v2.0) para extrair texto estruturado, e aplica
-- ai_extract (v2.1) com schema JSON para classificar tipo de documento,
-- resumo, alterações metodológicas e arquivos descritos. Salva como tabela
-- Bronze para consulta no Catalog Explorer.
-- ============================================================================
CREATE OR REPLACE TABLE workspace.bacen_bronze.documentacao_pdfs AS
WITH pdfs AS (
  SELECT
    _metadata.file_name AS file_name,
    path,
    content,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '/Volumes/workspace/bacen_bronze/bacen_reclamacoes_site/2026/',
    format => 'binaryFile',
    fileNamePattern => '*.pdf'
  )
),
extracao AS (
  SELECT
    file_name,
    path,
    ai_extract(
      parsed,
      '{
        "tipo_documento": {"type": "string", "description": "Tipo: histórico de alterações, esclarecimentos mensal, ou esclarecimentos semestral"},
        "resumo": {"type": "string", "description": "Resumo conciso do conteúdo do documento"},
        "alteracoes": {
          "type": "array",
          "description": "Lista de alterações na tabela de irregularidades (quando aplicável)",
          "items": {
            "type": "object",
            "properties": {
              "data": {"type": "string", "description": "Data ou período da alteração"},
              "descricao": {"type": "string", "description": "Descrição da alteração"}
            }
          }
        },
        "metodologia": {"type": "string", "description": "Descrição da metodologia de cálculo ou classificação"},
        "arquivos_descritos": {
          "type": "array",
          "description": "Arquivos descritos no documento de esclarecimentos",
          "items": {
            "type": "object",
            "properties": {
              "nome": {"type": "string"},
              "descricao": {"type": "string"},
              "periodicidade": {"type": "string"}
            }
          }
        }
      }',
      MAP('version', '2.1', 'instructions', 'Documentos do BACEN sobre ranking de reclamações. Extrair informações sobre metodologia, alterações na tabela de irregularidades e descrições dos arquivos de download.')
    ) AS dados_extraidos,
    parsed:metadata AS metadata,
    is_variant_null(parsed:error_status) AS parse_ok
  FROM pdfs
)
SELECT
  file_name,
  path,
  dados_extraidos:tipo_documento AS tipo_documento,
  dados_extraidos:resumo AS resumo,
  dados_extraidos:alteracoes AS alteracoes,
  dados_extraidos:metodologia AS metodologia,
  dados_extraidos:arquivos_descritos AS arquivos_descritos,
  parse_ok,
  metadata
FROM extracao

In [0]:
# ============================================================================
# VERIFICAÇÃO DA EXTRAÇÃO DE PDFs
# ============================================================================
# Verifica a tabela documentacao_pdfs: contagem de PDFs processados,
# listagem por tipo de documento com status de parse, e resumos extraídos.
# Valida que o ai_parse_document + ai_extract funcionaram corretamente.
# ============================================================================

print("📊 Tabela: workspace.bacen_bronze.documentacao_pdfs")
count = spark.sql("SELECT COUNT(*) FROM workspace.bacen_bronze.documentacao_pdfs").collect()[0][0]
print(f"   {count} PDFs processados")

print("\n📋 Documentos por tipo:")
spark.sql("""
SELECT file_name, tipo_documento, parse_ok
FROM workspace.bacen_bronze.documentacao_pdfs
ORDER BY file_name
""").show(truncate=50)

print("\n📝 Resumos extraídos:")
spark.sql("""
SELECT file_name, tipo_documento, resumo
FROM workspace.bacen_bronze.documentacao_pdfs
WHERE resumo IS NOT NULL
ORDER BY file_name
""").show(truncate=80)